In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import ipywidgets as widgets

from IPython.display import display
import StepMiner

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.dpi"] = 150

# change only if needed
BASE_DIR = "./"
RAW_DIR = os.path.join(BASE_DIR, "RAW")
OUT_DIR = os.path.join(BASE_DIR, "interactive_compass_score_plots")
os.makedirs(OUT_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR exists:", os.path.exists(RAW_DIR))
print("OUT_DIR:", OUT_DIR)

In [ ]:
def list_samples():
    expr_files = sorted(glob.glob(os.path.join(RAW_DIR, "*_exprMat_file.csv.gz")))
    return [os.path.basename(x).replace("_exprMat_file.csv.gz", "") for x in expr_files]


def image_folder_name(sample_name):
    # GSM7473684_HC_c -> GSM7473684_HCc
    parts = sample_name.split("_")
    gsm = parts[0]
    disease = parts[1]
    rep = parts[2]
    return f"{gsm}_{disease}{rep}"


def expr_file_path(sample_name):
    p = os.path.join(RAW_DIR, f"{sample_name}_exprMat_file.csv.gz")
    if os.path.exists(p):
        return p
    raise FileNotFoundError(f"exprMat file not found: {p}")


def meta_file_path(sample_name):
    p = os.path.join(RAW_DIR, f"{sample_name}_metadata_file.csv.gz")
    if os.path.exists(p):
        return p
    raise FileNotFoundError(f"metadata file not found: {p}")


def find_sample_dir(sample_name):
    p = os.path.join(RAW_DIR, image_folder_name(sample_name))
    if os.path.isdir(p):
        return p
    raise FileNotFoundError(f"sample image directory not found: {p}")


def composite_path(sample_name, fov):
    sample_dir = find_sample_dir(sample_name)

    patterns = [
        os.path.join(sample_dir, "CellComposite", f"CellComposite_F{int(fov):03d}.jpg"),
        os.path.join(sample_dir, "CellComposite", f"CellComposite_F{int(fov):03d}.jpeg"),
        os.path.join(sample_dir, "CellComposite", f"CellComposite_F{int(fov):03d}.png"),
        os.path.join(sample_dir, "CellComposite", f"*F{int(fov):03d}*"),
    ]

    for pat in patterns:
        matches = sorted(glob.glob(pat))
        if matches:
            return matches[0]

    raise FileNotFoundError(f"No CellComposite image found for sample={sample_name}, fov={fov}")

In [ ]:
def get_available_fovs(sample_name):
    expr = pd.read_csv(expr_file_path(sample_name), compression="gzip", usecols=["fov"])
    return sorted(expr["fov"].dropna().unique().tolist())


def get_available_genes(sample_name):
    expr = pd.read_csv(expr_file_path(sample_name), compression="gzip", nrows=2)
    cols = expr.columns.tolist()
    genes = [c for c in cols if c not in ["fov", "cell_ID"]]
    return genes


def load_expr_meta(sample_name):
    expr = pd.read_csv(expr_file_path(sample_name), compression="gzip")
    meta = pd.read_csv(meta_file_path(sample_name), compression="gzip")

    required_meta = ["fov", "cell_ID", "CenterX_local_px", "CenterY_local_px"]
    missing = [c for c in required_meta if c not in meta.columns]
    if missing:
        raise ValueError(f"Missing required metadata columns: {missing}")

    merged = expr.merge(meta, on=["fov", "cell_ID"], how="inner")
    merged = merged.rename(columns={
        "CenterX_local_px": "x_local",
        "CenterY_local_px": "y_local"
    })
    return merged


def to_grayscale(img):
    if img.ndim == 2:
        return img
    if img.shape[2] == 4:
        img = img[:, :, :3]
    return np.dot(img[..., :3], [0.2989, 0.5870, 0.1140])

In [ ]:
def transform_coords(x, y, img_shape, mode="flip_y"):
    h, w = img_shape[:2]
    x = np.asarray(x).copy()
    y = np.asarray(y).copy()

    if mode == "none":
        return x, y
    elif mode == "flip_x":
        return (w - x), y
    elif mode == "flip_y":
        return x, (h - y)
    elif mode == "flip_xy":
        return (w - x), (h - y)
    elif mode == "rot90":
        return (h - y), x
    elif mode == "rot180":
        return (w - x), (h - y)
    elif mode == "rot270":
        return y, (w - x)
    else:
        raise ValueError(f"Unknown transform mode: {mode}")

In [ ]:
def stepminer_threshold(values):
    arr = np.array([float(x) for x in values if pd.notnull(x)], dtype=float)
    arr.sort()
    if arr.size == 0:
        return np.nan
    if arr.size < 3:
        return float(np.median(arr))
    s = StepMiner.fitstep(arr)
    return float(s["threshold"])


def compute_rank_vector_cells(expr_df_cells_by_genes, genes):
    """
    expr_df_cells_by_genes: rows = cells, cols = genes
    returns one score per cell
    """
    acc = np.zeros(expr_df_cells_by_genes.shape[0], dtype=float)

    for g_raw in genes:
        g = str(g_raw).upper()
        cols_upper = {c.upper(): c for c in expr_df_cells_by_genes.columns}
        if g not in cols_upper:
            continue

        col = cols_upper[g]
        v = expr_df_cells_by_genes[col].astype(float).values

        sd = np.std(v)
        if not np.isfinite(sd) or sd <= 0:
            continue

        thr = stepminer_threshold(v)
        if not np.isfinite(thr):
            continue

        t3 = thr + 0.5
        score = (v - t3) / (3.0 * sd)
        bad = ~np.isfinite(score)
        if bad.any():
            score[bad] = 0.0

        acc += score

    return pd.Series(acc, index=expr_df_cells_by_genes.index)


def compute_compass_cells(expr_df_cells_by_genes, modules, weights):
    if len(modules) == 0:
        raise ValueError("No gene modules provided.")
    if len(weights) != len(modules):
        raise ValueError("Weights must match number of modules.")

    rv_list = [compute_rank_vector_cells(expr_df_cells_by_genes, mod) for mod in modules]
    comp = np.zeros(len(rv_list[0]), dtype=float)

    for w, rv in zip(weights, rv_list):
        comp += float(w) * rv.values

    return pd.Series(comp, index=rv_list[0].index, name="CompositeScore")

In [ ]:
def plot_compass_score_overlay(
    sample_name,
    fov,
    modules,
    weights,
    cmap="inferno",
    point_size=8,
    alpha=0.9,
    crop_quantile=0.02,
    pad_fraction=0.15,
    min_pad_px=100,
    panel_width=7,
    background_mode="gray",
    background_alpha=1.0,
    transform_mode="flip_y",
    clip_quantile=0.995,
    save_pdf=False,
    output_prefix=None,
    show_title=True
):
    img_path = composite_path(sample_name, fov)
    img = mpimg.imread(img_path)

    if background_mode == "gray":
        bg = to_grayscale(img)
        bg_cmap = "gray"
    else:
        bg = img
        bg_cmap = None

    df = load_expr_meta(sample_name)
    df_fov = df[df["fov"] == int(fov)].copy()

    if df_fov.empty:
        raise ValueError(f"No cells found for sample={sample_name}, fov={fov}")

    # expression matrix for scoring
    exclude_cols = {
        "fov", "cell_ID", "x_local", "y_local",
        "Area", "AspectRatio", "CenterX_global_px", "CenterY_global_px",
        "Width", "Height"
    }
    gene_cols = [c for c in df_fov.columns if c not in exclude_cols]
    expr_cells = df_fov[gene_cols].copy()

    # compute COMPASS score
    comp = compute_compass_cells(expr_cells, modules, weights)
    df_fov["CompositeScore"] = comp.values

    # transform coordinates
    df_fov["x_plot"], df_fov["y_plot"] = transform_coords(
        df_fov["x_local"].values,
        df_fov["y_local"].values,
        bg.shape,
        mode=transform_mode
    )

    # crop from all cells in FOV, not just high-score cells
    x0 = df_fov["x_plot"].quantile(crop_quantile)
    x1 = df_fov["x_plot"].quantile(1 - crop_quantile)
    y0 = df_fov["y_plot"].quantile(crop_quantile)
    y1 = df_fov["y_plot"].quantile(1 - crop_quantile)

    pad_x = max((x1 - x0) * pad_fraction, min_pad_px)
    pad_y = max((y1 - y0) * pad_fraction, min_pad_px)

    xmin = max(0, x0 - pad_x)
    xmax = min(bg.shape[1], x1 + pad_x)
    ymin = max(0, y0 - pad_y)
    ymax = min(bg.shape[0], y1 + pad_y)

    # score clipping for visual robustness
    vmin = df_fov["CompositeScore"].quantile(1 - clip_quantile)
    vmax = df_fov["CompositeScore"].quantile(clip_quantile)

    fig, ax = plt.subplots(figsize=(panel_width, panel_width))

    if background_mode == "gray":
        ax.imshow(bg, cmap=bg_cmap, alpha=background_alpha)
    else:
        ax.imshow(bg, alpha=background_alpha)

    sc = ax.scatter(
        df_fov["x_plot"],
        df_fov["y_plot"],
        c=df_fov["CompositeScore"],
        s=point_size,
        cmap=cmap,
        alpha=alpha,
        linewidths=0,
        vmin=vmin,
        vmax=vmax
    )

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymax, ymin)
    if show_title:
        ax.set_title(f"{sample_name} | FoV {fov} | COMPASS score", fontsize=14)
    ax.axis("off")

    cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Composite score")

    plt.tight_layout()

    if save_pdf:
        if output_prefix is None:
            output_prefix = f"{sample_name}_FOV{fov}_COMPASS_score"
        out_pdf = os.path.join(OUT_DIR, f"{output_prefix}.pdf")
        plt.savefig(out_pdf, dpi=300, bbox_inches="tight")
        print("Saved:", out_pdf)

    plt.show()
    plt.close()

    return df_fov

In [ ]:
sample_name = "GSM7473684_HC_c"
fov = 1

modules = [
    ["GENE_A", "GENE_B"],   # module 1
    ["GENE_C"]             # module 2
]
weights = [1.0, -1.0]

df_scored = plot_compass_score_overlay(
    sample_name=sample_name,
    fov=fov,
    modules=modules,
    weights=weights,
    cmap="inferno",
    point_size=8,
    alpha=0.9,
    background_mode="gray",
    transform_mode="flip_y",
    save_pdf=False
)

df_scored.head()

In [ ]:
samples = list_samples()

sample_dropdown = widgets.Dropdown(
    options=samples,
    description="Sample:",
    value=samples[0] if samples else None,
    layout=widgets.Layout(width="450px")
)

fov_dropdown = widgets.Dropdown(
    options=[],
    description="FoV:",
    layout=widgets.Layout(width="250px")
)

module_count = widgets.IntSlider(
    value=2,
    min=1,
    max=6,
    step=1,
    description="N modules:"
)

module_gene_boxes = []
module_weight_boxes = []

for i in range(6):
    gene_box = widgets.Textarea(
        value="",
        placeholder="Gene1, Gene2, Gene3",
        description=f"M{i+1}",
        layout=widgets.Layout(width="420px", height="70px")
    )
    weight_box = widgets.FloatText(
        value=1.0,
        description="Weight",
        layout=widgets.Layout(width="170px")
    )
    module_gene_boxes.append(gene_box)
    module_weight_boxes.append(weight_box)

module_box = widgets.VBox()

point_size_slider = widgets.FloatSlider(
    value=8,
    min=1,
    max=30,
    step=1,
    description="Point size:"
)

alpha_slider = widgets.FloatSlider(
    value=0.9,
    min=0.1,
    max=1.0,
    step=0.05,
    description="Alpha:"
)

crop_quantile_slider = widgets.FloatSlider(
    value=0.02,
    min=0.0,
    max=0.2,
    step=0.01,
    description="Crop q:"
)

pad_fraction_slider = widgets.FloatSlider(
    value=0.15,
    min=0.0,
    max=0.5,
    step=0.01,
    description="Pad frac:"
)

min_pad_slider = widgets.IntSlider(
    value=100,
    min=0,
    max=500,
    step=10,
    description="Min pad:"
)

panel_width_slider = widgets.FloatSlider(
    value=7,
    min=4,
    max=12,
    step=0.5,
    description="Panel w:"
)

background_dropdown = widgets.Dropdown(
    options=["gray", "color"],
    value="gray",
    description="BG mode:"
)

background_alpha_slider = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=1.0,
    step=0.05,
    description="BG alpha:"
)

transform_dropdown = widgets.Dropdown(
    options=["flip_y", "none", "flip_x", "flip_xy", "rot90", "rot180", "rot270"],
    value="flip_y",
    description="Transform:"
)

cmap_dropdown = widgets.Dropdown(
    options=["inferno", "viridis", "plasma", "magma", "coolwarm", "RdBu_r"],
    value="coolwarm",
    description="Colormap:"
)

clip_quantile_slider = widgets.FloatSlider(
    value=0.995,
    min=0.80,
    max=1.0,
    step=0.005,
    description="Clip q:"
)

save_checkbox = widgets.Checkbox(
    value=False,
    description="Save PDF"
)

output_text = widgets.Text(
    value="",
    description="Out name:",
    layout=widgets.Layout(width="500px")
)

plot_button = widgets.Button(
    description="Compute + Plot",
    button_style="success"
)

status_out = widgets.Output()
plot_out = widgets.Output()

In [ ]:
def update_module_box(*args):
    n = module_count.value
    module_box.children = [
        widgets.HBox([module_gene_boxes[i], module_weight_boxes[i]])
        for i in range(n)
    ]

module_count.observe(update_module_box, names="value")
update_module_box()

In [ ]:
def update_sample_options(*args):
    sample = sample_dropdown.value
    if sample is None:
        return

    try:
        fovs = get_available_fovs(sample)
        fov_dropdown.options = fovs
        if len(fovs) > 0:
            fov_dropdown.value = fovs[0]

        with status_out:
            status_out.clear_output()
            print(f"Loaded sample: {sample}")
            print(f"FoVs: {len(fovs)}")

    except Exception as e:
        with status_out:
            status_out.clear_output()
            print("Error loading sample options:")
            print(e)

sample_dropdown.observe(update_sample_options, names="value")
update_sample_options()

In [ ]:
def parse_module_text(text):
    genes = [g.strip().upper() for g in text.replace("\n", " ").replace(",", " ").split() if g.strip()]
    return genes

def on_plot_clicked(b):
    with plot_out:
        plot_out.clear_output()

        try:
            n = module_count.value
            modules = []
            weights = []

            for i in range(n):
                genes = parse_module_text(module_gene_boxes[i].value)
                wt = float(module_weight_boxes[i].value)

                if len(genes) > 0:
                    modules.append(genes)
                    weights.append(wt)

            if len(modules) == 0:
                print("Please enter at least one non-empty gene module.")
                return

            df_scored = plot_compass_score_overlay(
                sample_name=sample_dropdown.value,
                fov=fov_dropdown.value,
                modules=modules,
                weights=weights,
                cmap=cmap_dropdown.value,
                point_size=point_size_slider.value,
                alpha=alpha_slider.value,
                crop_quantile=crop_quantile_slider.value,
                pad_fraction=pad_fraction_slider.value,
                min_pad_px=min_pad_slider.value,
                panel_width=panel_width_slider.value,
                background_mode=background_dropdown.value,
                background_alpha=background_alpha_slider.value,
                transform_mode=transform_dropdown.value,
                clip_quantile=clip_quantile_slider.value,
                save_pdf=save_checkbox.value,
                output_prefix=output_text.value if output_text.value.strip() else None,
                show_title=True
            )

            print("\nComposite score summary:")
            print(df_scored["CompositeScore"].describe())

        except Exception as e:
            print("Plotting error:")
            print(e)

plot_button.on_click(on_plot_clicked)

In [ ]:
controls_left = widgets.VBox([
    sample_dropdown,
    fov_dropdown,
    module_count,
    module_box
])

controls_mid = widgets.VBox([
    point_size_slider,
    alpha_slider,
    crop_quantile_slider,
    pad_fraction_slider,
    min_pad_slider,
    panel_width_slider
])

controls_right = widgets.VBox([
    background_dropdown,
    background_alpha_slider,
    transform_dropdown,
    cmap_dropdown,
    clip_quantile_slider,
    save_checkbox,
    output_text,
    plot_button
])

ui = widgets.HBox([controls_left, controls_mid, controls_right])

display(ui)
display(status_out)
display(plot_out)
print ('working')
